# Module 2b: Strands-Evals Evaluation for Single Agent

## Overview

This notebook evaluates the single-agent product catalog system from Module 1 using **strands-evals**.

### Optimization: Run Agent Once, Evaluate Multiple Times

Unlike the naive approach that runs the agent once per evaluator (6x per test case), this notebook:
1. **Runs the agent ONCE** per test case and caches responses
2. **Evaluates cached responses** with all metrics

This reduces agent invocations from `N_cases × N_evaluators` to just `N_cases`.

### Evaluation Metrics (6 total)
1. Goal Success
2. Helpfulness
3. Tool Usage Accuracy (adapted for single agent)
4. Policy Compliance
5. Response Quality
6. Customer Satisfaction

### Time: ~30 minutes

## Step 1: Setup

Install dependencies and create the customer service instance.

In [1]:
# Install dependencies
!pip install -q strands-agents strands-agents-evals boto3 pandas deepeval

In [4]:
import json
import os
import sys
import boto3
import pandas as pd
from datetime import datetime

# Try to load REGION from Module 1
try:
    %store -r REGION
    print(f"✓ Loaded REGION from Module 1: {REGION}")
except:
    print("Could not load REGION from Module 1, using session default")
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'

print(f"Region: {REGION}")

# Set environment variable
os.environ['AWS_REGION'] = REGION

# Import Product Catalog Agent from Module 1
print("\nImporting Product Catalog Agent...")
sys.path.insert(0, '../01-single-agent-prototype/agents')
from product_catalog_agent import ProductCatalogAgent, UserSession
print("✓ Product Catalog Agent imported successfully")

✓ Loaded REGION from Module 1: us-west-2
Region: us-west-2

Importing Product Catalog Agent...
✓ Product Catalog Agent imported successfully


## Step 2: Load Evaluation Experiment

Load the test cases and convert them to the format expected by strands-evals.

In [5]:
# Import strands-evals
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

# Load evaluation dataset
with open('evaluation_dataset.json', 'r') as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data['test_cases'])} test cases")
print(f"\nSample test case:")
print(json.dumps(eval_data['test_cases'][0], indent=2))

Loaded 25 test cases

Sample test case:
{
  "id": "order_001",
  "category": "order_status",
  "difficulty": "easy",
  "input": "What's the status of order ORD-2024-10002?",
  "expected_agent": "order_agent",
  "expected_tools": [
    "get_order_status"
  ],
  "expected_output_contains": [
    "shipped",
    "UPS",
    "tracking"
  ],
  "ground_truth": "Order ORD-2024-10002 is shipped via UPS with tracking number 1Z999AA10123456784"
}


In [6]:
# Convert test cases to Case objects
test_cases = []
for tc in eval_data['test_cases']:
    case = Case(
        name=tc['id'],
        input=tc['input'],
        expected_output=tc.get('ground_truth', ''),
        metadata={
            'category': tc['category'],
            'expected_agent': tc.get('expected_agent'),
            'expected_tools': tc.get('expected_tools', []),
            'expected_output_contains': tc.get('expected_output_contains', [])
        }
    )
    test_cases.append(case)

print(f"Created {len(test_cases)} Case objects")
print(f"\nCategories: {set(tc.metadata['category'] for tc in test_cases)}")

Created 25 Case objects

Categories: {'product_comparison', 'multi_agent', 'product_policy', 'suspended_account', 'membership_benefits', 'product_inventory', 'edge_case', 'out_of_scope', 'order_status', 'order_cancellation', 'account_info', 'order_return', 'password_reset', 'order_tracking', 'address_update', 'order_not_found', 'product_recommendation', 'product_details', 'payment_methods', 'product_search', 'order_history', 'refund_status'}


## Step 3: Run Agent Once and Cache Responses

**Key Optimization**: Run the agent ONCE for each test case and cache responses. This avoids running the agent 6x per test case (once per evaluator).

In [7]:
import time

# Select 5 product-related test cases for single-agent evaluation
# (Module 1 only handles product catalog queries)
SELECTED_IDS = [
    'product_001',    # Product search: wireless headphones
    'product_002',    # Inventory check: webcam stock  
    'product_003',    # Product recommendations: accessories
    'product_004',    # Product comparison: headphones vs earbuds
    'product_005',    # Product details: specific product
]

# Filter test cases by selected IDs
selected_cases = [tc for tc in test_cases if tc.name in SELECTED_IDS]

print(f"Selected {len(selected_cases)} product-related test cases:")
for tc in selected_cases:
    print(f"  - {tc.name} ({tc.metadata['category']}): {tc.input[:50]}...")

# Cache to store agent responses (key: case name, value: response)
response_cache = {}

def run_agent_and_cache(cases: list, region: str) -> dict:
    """
    Run the agent ONCE for each test case and cache responses.
    
    Args:
        cases: List of Case objects to evaluate
        region: AWS region for agent initialization
        
    Returns:
        dict: Mapping of case name to response
    """
    print(f"\nRunning agent on {len(cases)} test cases (ONE TIME ONLY)...\n")
    
    for i, case in enumerate(cases):
        print(f"[{i+1}/{len(cases)}] {case.name} ({case.metadata['category']})")
        print(f"    Query: {case.input[:60]}...")
        
        # Create test session with customer role
        session = UserSession(
            user_id="test_customer",
            role="customer",
            email="test@example.com",
            name="Test User"
        )
        
        start_time = time.time()
        try:
            # Initialize agent for this test case
            agent = ProductCatalogAgent(region=region, user_session=session)
            
            # Run the agent
            response = agent(case.input)
            latency = time.time() - start_time
            response_cache[case.name] = str(response)
            print(f"    Latency: {latency:.1f}s")
            print(f"    Response: {str(response)[:100]}...")
            
            # Clean up agent
            agent.cleanup()
            
        except Exception as e:
            response_cache[case.name] = f"Error: {str(e)}"
            print(f"    ERROR: {str(e)[:50]}")
        print()
        time.sleep(0.5)  # Rate limiting
    
    return response_cache


def cached_task(case) -> str:
    """
    Task function that returns cached response instead of calling agent.
    This allows running multiple evaluators without re-invoking the agent.
    """
    return response_cache.get(case.name, "Error: Response not found in cache")


# Run agent on selected product test cases and cache responses
response_cache = run_agent_and_cache(selected_cases, REGION)

print(f"\n✓ Cached {len(response_cache)} responses")
print("✓ All evaluators will use cached responses (no additional agent calls)")

Selected 5 product-related test cases:
  - product_001 (product_search): Do you have any wireless headphones with noise can...
  - product_002 (product_inventory): Is the 4K webcam PROD-088 in stock?...
  - product_003 (product_recommendation): I just bought wireless headphones. What accessorie...
  - product_004 (product_comparison): Compare the Wireless Bluetooth Headphones PROD-001...
  - product_005 (product_policy): What's your return policy for electronics?...

Running agent on 5 test cases (ONE TIME ONLY)...

[1/5] product_001 (product_search)
    Query: Do you have any wireless headphones with noise cancellation?...
I'll search for wireless headphones with noise cancellation for you.
Tool #1: search_products
Great! I found several options for you. Here are the wireless headphones with noise cancellation we have available:

**1. Wireless Bluetooth Headphones (PROD-001)** ✓ In Stock
- **Price:** $79.99
- **Type:** Over-ear
- **Key Features:**
  - Active noise cancellation
  - 40-

## Step 4: Run Evaluators on Cached Responses

Now run all 6 evaluators using the **cached responses**. No additional agent calls are made.

**Note:** Each Experiment uses `cached_task` which returns pre-computed responses.

In [8]:
# Create Goal Success Evaluator
goal_success_evaluator = OutputEvaluator(
    rubric="""Evaluate if the agent response successfully addresses the customer's request.

Score 1.0: The response fully addresses the customer's request with accurate, helpful information
Score 0.75: The response mostly addresses the request but may be missing minor details
Score 0.5: The response partially addresses the request but has significant gaps
Score 0.25: The response attempts to address the request but fails to provide useful help
Score 0.0: The response does not address the request at all or provides incorrect information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

print("Goal Success Evaluator created")

Goal Success Evaluator created


In [9]:
# Run Goal Success Evaluation using CACHED responses
goal_success_experiment = Experiment(
    cases=selected_cases,
    evaluators=[goal_success_evaluator]
)

print("Running goal success evaluation (using cached responses)...")
goal_success_results = goal_success_experiment.run_evaluations(cached_task)

Running goal success evaluation (using cached responses)...


In [10]:
# Extract the report (first and only item in results list)
goal_success_report = goal_success_results[0]

print("\n" + "="*60)
print("GOAL SUCCESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, goal_success_report.scores, goal_success_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Goal Success Score: {goal_success_report.overall_score:.2f}")
print(f"Pass Rate: {sum(goal_success_report.test_passes)}/{len(goal_success_report.test_passes)}")
print(f"{'='*60}")


GOAL SUCCESS EVALUATION RESULTS

1. product_001 (product_search)
   Score: 1.00
   Reasoning: The response fully addresses the customer's request by finding and presenting PROD-001 Wireless Bluetooth Headphones with active noise cancellation as...

2. product_002 (product_inventory)
   Score: 0.00
   Reasoning: The output provides completely incorrect information. It states the product is in stock with 100 units available, when the expected output indicates i...

3. product_003 (product_recommendation)
   Score: 0.75
   Reasoning: The response addresses the customer's request for accessories by recommending complementary products like cases, stands, and cables as mentioned in th...

4. product_004 (product_comparison)
   Score: 1.00
   Reasoning: The output fully addresses the customer's request by providing a comprehensive comparison between the two products. It includes accurate pricing ($79....

5. product_005 (product_policy)
   Score: 1.00
   Reasoning: The output fully addresses

In [11]:
# Create Helpfulness Evaluator
helpfulness_evaluator = OutputEvaluator(
    rubric="""Evaluate how helpful the agent response is for the customer.

Score 1.0: Extremely helpful - provides clear, actionable information and anticipates follow-up needs
Score 0.75: Very helpful - provides good information that addresses the customer's needs
Score 0.5: Somewhat helpful - provides basic information but could be more detailed
Score 0.25: Minimally helpful - provides limited useful information
Score 0.0: Not helpful - does not provide any useful information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

# Run Helpfulness Evaluation using CACHED responses
helpfulness_experiment = Experiment(
    cases=selected_cases,
    evaluators=[helpfulness_evaluator]
)

print("Running helpfulness evaluation (using cached responses)...")
helpfulness_results = helpfulness_experiment.run_evaluations(cached_task)

# Extract the report
helpfulness_report = helpfulness_results[0]

print("\n" + "="*60)
print("HELPFULNESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, helpfulness_report.scores, helpfulness_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Helpfulness Score: {helpfulness_report.overall_score:.2f}")
print(f"Pass Rate: {sum(helpfulness_report.test_passes)}/{len(helpfulness_report.test_passes)}")
print(f"{'='*60}")

Running helpfulness evaluation (using cached responses)...

HELPFULNESS EVALUATION RESULTS

1. product_001 (product_search)
   Score: 1.00
   Reasoning: The response is extremely helpful. It not only finds the expected PROD-001 Wireless Bluetooth Headphones with ANC, but goes above and beyond by provid...

2. product_002 (product_inventory)
   Score: 0.00
   Reasoning: The output provides completely incorrect information, stating the product is in stock with 100 units available when it's actually out of stock. This m...

3. product_003 (product_recommendation)
   Score: 0.75
   Reasoning: The output is very helpful as it provides comprehensive recommendations including complementary products like cases and stands (as expected), plus add...

4. product_004 (product_comparison)
   Score: 1.00
   Reasoning: The response is extremely helpful - it provides a comprehensive comparison with detailed specifications, clear pricing, pros/cons analysis, and person...

5. product_005 (product_polic

## Step 5: Custom Evaluators (Using Cached Responses)

Run domain-specific custom evaluators. All evaluators use the same cached responses.

In [12]:
# Reload custom evaluators module (in case it was already imported)
import importlib
import sys
if 'custom_evaluators' in sys.modules:
    import custom_evaluators
    importlib.reload(custom_evaluators)

In [13]:
# Import custom evaluators
from custom_evaluators import (
    RoutingAccuracyEvaluator,
    PolicyComplianceEvaluator,
    ResponseQualityEvaluator,
    CustomerSatisfactionEvaluator
)

print("Custom evaluators imported")

Custom evaluators imported


In [14]:
# Routing Accuracy Evaluation using CACHED responses
# Note: For single-agent, this evaluates whether product queries are handled appropriately
routing_evaluator = RoutingAccuracyEvaluator()
routing_experiment = Experiment(
    cases=selected_cases,
    evaluators=[routing_evaluator]
)

print("Running tool usage accuracy evaluation (using cached responses)...")
print("(Adapted for single-agent: evaluates appropriate tool selection)")
routing_results = routing_experiment.run_evaluations(cached_task)

# Extract the report
routing_report = routing_results[0]

print("\n" + "="*60)
print("TOOL USAGE ACCURACY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, routing_report.scores, routing_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Tool Usage Accuracy Score: {routing_report.overall_score:.2f}")
print(f"Pass Rate: {sum(routing_report.test_passes)}/{len(routing_report.test_passes)}")
print(f"{'='*60}")

Running tool usage accuracy evaluation (using cached responses)...
(Adapted for single-agent: evaluates appropriate tool selection)

TOOL USAGE ACCURACY EVALUATION RESULTS

1. product_001 (product_search)
   Score: 1.00
   Reasoning: Perfect routing. The customer asked for wireless headphones with noise cancellation, which is clearly a product search/recommendation request. The res...

2. product_002 (product_inventory)
   Score: 1.00
   Reasoning: Perfect routing - the inventory inquiry was correctly directed to the Product Agent, who handles inventory requests and provided appropriate product i...

3. product_003 (product_recommendation)
   Score: 1.00
   Reasoning: Perfect routing - customer requested product recommendations for accessories, which is exactly what the Product Agent specializes in. The response dem...

4. product_004 (product_comparison)
   Score: 1.00
   Reasoning: Perfect routing - the customer requested a product comparison between two specific items (PROD-001 and 

In [15]:
# Policy Compliance Evaluation using CACHED responses
policy_evaluator = PolicyComplianceEvaluator()
policy_experiment = Experiment(
    cases=selected_cases,
    evaluators=[policy_evaluator]
)

print("Running policy compliance evaluation (using cached responses)...")
policy_results = policy_experiment.run_evaluations(cached_task)

# Extract the report
policy_report = policy_results[0]

print("\n" + "="*60)
print("POLICY COMPLIANCE EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, policy_report.scores, policy_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Policy Compliance Score: {policy_report.overall_score:.2f}")
print(f"Pass Rate: {sum(policy_report.test_passes)}/{len(policy_report.test_passes)}")
print(f"{'='*60}")

Running policy compliance evaluation (using cached responses)...

POLICY COMPLIANCE EVALUATION RESULTS

1. product_001 (product_search)
   Score: 1.00
   Reasoning: The output successfully found PROD-001 Wireless Bluetooth Headphones with noise cancellation as expected. The response doesn't involve any policy-rela...

2. product_002 (product_inventory)
   Score: 0.00
   Reasoning: The output provides completely incorrect inventory information, stating the item is in stock with 100 units available when the expected output shows i...

3. product_003 (product_recommendation)
   Score: 1.00
   Reasoning: The output is fully compliant with company policies. It provides product recommendations without making any policy-related statements that could viola...

4. product_004 (product_comparison)
   Score: 1.00
   Reasoning: The response is fully compliant as it focuses solely on product comparison without addressing any policy areas. No company policies were violated or i...

5. product_005 (p

In [16]:
# Response Quality Evaluation using CACHED responses
quality_evaluator = ResponseQualityEvaluator()
quality_experiment = Experiment(
    cases=selected_cases,
    evaluators=[quality_evaluator]
)

print("Running response quality evaluation (using cached responses)...")
quality_results = quality_experiment.run_evaluations(cached_task)

# Extract the report
quality_report = quality_results[0]

print("\n" + "="*60)
print("RESPONSE QUALITY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, quality_report.scores, quality_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Response Quality Score: {quality_report.overall_score:.2f}")
print(f"Pass Rate: {sum(quality_report.test_passes)}/{len(quality_report.test_passes)}")
print(f"{'='*60}")

Running response quality evaluation (using cached responses)...

RESPONSE QUALITY EVALUATION RESULTS

1. product_001 (product_search)
   Score: 0.80
   Reasoning: The response successfully identifies PROD-001 Wireless Bluetooth Headphones with ANC as expected, plus provides additional relevant options. It's high...

2. product_002 (product_inventory)
   Score: 0.00
   Reasoning: The response provides completely incorrect information. It states the product is in stock with 100 units available, when the expected output shows it'...

3. product_003 (product_recommendation)
   Score: 0.40
   Reasoning: The response misunderstands the customer's request for accessories for their existing wireless headphones. Instead of recommending actual accessories ...

4. product_004 (product_comparison)
   Score: 1.00
   Reasoning: The response excellently addresses the customer's product comparison request. It provides a comprehensive, well-structured comparison with detailed sp...

5. product_005 (pro

In [17]:
# Customer Satisfaction Evaluation using CACHED responses
satisfaction_evaluator = CustomerSatisfactionEvaluator()
satisfaction_experiment = Experiment(
    cases=selected_cases,
    evaluators=[satisfaction_evaluator]
)

print("Running customer satisfaction evaluation (using cached responses)...")
satisfaction_results = satisfaction_experiment.run_evaluations(cached_task)

# Extract the report
satisfaction_report = satisfaction_results[0]

print("\n" + "="*60)
print("CUSTOMER SATISFACTION EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, satisfaction_report.scores, satisfaction_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Customer Satisfaction Score: {satisfaction_report.overall_score:.2f}")
print(f"Pass Rate: {sum(satisfaction_report.test_passes)}/{len(satisfaction_report.test_passes)}")
print(f"{'='*60}")

Running customer satisfaction evaluation (using cached responses)...

CUSTOMER SATISFACTION EVALUATION RESULTS

1. product_001 (product_search)
   Score: 1.00
   Reasoning: The customer's primary issue was fully resolved - they asked for wireless headphones with noise cancellation and received detailed information about m...

2. product_002 (product_inventory)
   Score: 0.00
   Reasoning: The output provides completely incorrect information - stating the product is in stock with 100 units available when it's actually out of stock. This ...

3. product_003 (product_recommendation)
   Score: 0.75
   Reasoning: The response addresses the customer's request for accessory recommendations and includes the expected complementary products (cases, stands, cables). ...

4. product_004 (product_comparison)
   Score: 1.00
   Reasoning: The customer requested a comparison between two specific audio products, and the response fully resolved their primary issue by providing a comprehens...

5. produc

## Step 6: Extract and Analyze Results

Collect scores from all evaluations and compute baseline metrics.

In [18]:
# Helper function to extract scores from EvaluationReport
def extract_scores_from_report(report):
    """Extract scores from evaluation report"""
    return report.scores

# Extract all scores
goal_success_scores = extract_scores_from_report(goal_success_report)
helpfulness_scores = extract_scores_from_report(helpfulness_report)
routing_scores = extract_scores_from_report(routing_report)
policy_scores = extract_scores_from_report(policy_report)
quality_scores = extract_scores_from_report(quality_report)
satisfaction_scores = extract_scores_from_report(satisfaction_report)

print("✓ Scores extracted from all evaluations")
print(f"\nGoal Success: {goal_success_scores}")
print(f"Helpfulness: {helpfulness_scores}")
print(f"Routing: {routing_scores}")
print(f"Policy: {policy_scores}")
print(f"Quality: {quality_scores}")
print(f"Satisfaction: {satisfaction_scores}")

✓ Scores extracted from all evaluations

Goal Success: [1.0, 0.0, 0.75, 1.0, 1.0]
Helpfulness: [1.0, 0.0, 0.75, 1.0, 1.0]
Routing: [1.0, 1.0, 1.0, 1.0, 1.0]
Policy: [1.0, 0.0, 1.0, 1.0, 0.8]
Quality: [0.8, 0.0, 0.4, 1.0, 0.8]
Satisfaction: [1.0, 0.0, 0.75, 1.0, 0.9]


In [19]:
# Create DataFrame with all results
results_df = pd.DataFrame({
    'test_case': [case.name for case in selected_cases],
    'category': [case.metadata['category'] for case in selected_cases],
    'goal_success': goal_success_scores,
    'helpfulness': helpfulness_scores,
    'tool_usage_accuracy': routing_scores,  # Routing evaluator adapted for single-agent tool usage
    'policy_compliance': policy_scores,
    'response_quality': quality_scores,
    'customer_satisfaction': satisfaction_scores
})

print("\nEvaluation Results DataFrame:")
print(results_df.to_string(index=False))


Evaluation Results DataFrame:
  test_case               category  goal_success  helpfulness  tool_usage_accuracy  policy_compliance  response_quality  customer_satisfaction
product_001         product_search          1.00         1.00                  1.0                1.0               0.8                   1.00
product_002      product_inventory          0.00         0.00                  1.0                0.0               0.0                   0.00
product_003 product_recommendation          0.75         0.75                  1.0                1.0               0.4                   0.75
product_004     product_comparison          1.00         1.00                  1.0                1.0               1.0                   1.00
product_005         product_policy          1.00         1.00                  1.0                0.8               0.8                   0.90


## Step 7: Baseline Metrics

Calculate average scores to establish baseline performance.

In [20]:
# Calculate baseline metrics
baseline_metrics = {
    'timestamp': datetime.now().isoformat(),
    'total_test_cases': len(results_df),
    'goal_success_rate': results_df['goal_success'].mean(),
    'helpfulness': results_df['helpfulness'].mean(),
    'tool_usage_accuracy': results_df['tool_usage_accuracy'].mean(),
    'policy_compliance': results_df['policy_compliance'].mean(),
    'response_quality': results_df['response_quality'].mean(),
    'customer_satisfaction': results_df['customer_satisfaction'].mean()
}

print("\n" + "="*60)
print("BASELINE METRICS")
print("="*60)
for metric, value in baseline_metrics.items():
    if isinstance(value, float) and value <= 1.0:
        print(f"{metric:.<40} {value:.1%}")
    else:
        print(f"{metric:.<40} {value}")


BASELINE METRICS
timestamp............................... 2026-02-23T15:59:18.306527
total_test_cases........................ 5
goal_success_rate....................... 75.0%
helpfulness............................. 75.0%
tool_usage_accuracy..................... 100.0%
policy_compliance....................... 76.0%
response_quality........................ 60.0%
customer_satisfaction................... 73.0%


In [21]:
# Performance by category
print("\n" + "="*60)
print("PERFORMANCE BY CATEGORY")
print("="*60)

category_metrics = results_df.groupby('category')[[
    'goal_success', 'helpfulness', 'tool_usage_accuracy', 
    'policy_compliance', 'response_quality', 'customer_satisfaction'
]].mean()

print(category_metrics.to_string())


PERFORMANCE BY CATEGORY
                        goal_success  helpfulness  tool_usage_accuracy  policy_compliance  response_quality  customer_satisfaction
category                                                                                                                          
product_comparison              1.00         1.00                  1.0                1.0               1.0                   1.00
product_inventory               0.00         0.00                  1.0                0.0               0.0                   0.00
product_policy                  1.00         1.00                  1.0                0.8               0.8                   0.90
product_recommendation          0.75         0.75                  1.0                1.0               0.4                   0.75
product_search                  1.00         1.00                  1.0                1.0               0.8                   1.00


## Step 8: Define Production Thresholds

Set alert thresholds based on baseline metrics (typically 80-90% of baseline).

In [22]:
# Define production thresholds
production_thresholds = {
    'goal_success_rate': 0.70,        # Alert if below 70%
    'helpfulness': 0.65,               # Alert if below 65%
    'tool_usage_accuracy': 0.75,      # Alert if below 75%
    'policy_compliance': 0.80,         # Alert if below 80%
    'response_quality': 0.65,          # Alert if below 65%
    'customer_satisfaction': 0.70      # Alert if below 70%
}

print("\n" + "="*60)
print("PRODUCTION THRESHOLDS")
print("="*60)
for metric, threshold in production_thresholds.items():
    print(f"{metric:.<40} {threshold:.0%}")


PRODUCTION THRESHOLDS
goal_success_rate....................... 70%
helpfulness............................. 65%
tool_usage_accuracy..................... 75%
policy_compliance....................... 80%
response_quality........................ 65%
customer_satisfaction................... 70%


In [23]:
# Check current performance against thresholds
print("\n" + "="*60)
print("THRESHOLD VALIDATION")
print("="*60)

all_pass = True
for metric, threshold in production_thresholds.items():
    current = baseline_metrics.get(metric, 0)
    passed = current >= threshold
    
    status = "✓ PASS" if passed else "✗ FAIL"
    
    if not passed:
        all_pass = False
    
    print(f"[{status}] {metric}: {current:.1%} (threshold: {threshold:.0%})")

print("\n" + "="*60)
if all_pass:
    print("✓ ALL THRESHOLDS PASSED - Ready for production!")
else:
    print("⚠ SOME THRESHOLDS FAILED - Review before production")
print("="*60)


THRESHOLD VALIDATION
[✓ PASS] goal_success_rate: 75.0% (threshold: 70%)
[✓ PASS] helpfulness: 75.0% (threshold: 65%)
[✓ PASS] tool_usage_accuracy: 100.0% (threshold: 75%)
[✗ FAIL] policy_compliance: 76.0% (threshold: 80%)
[✗ FAIL] response_quality: 60.0% (threshold: 65%)
[✓ PASS] customer_satisfaction: 73.0% (threshold: 70%)

⚠ SOME THRESHOLDS FAILED - Review before production


## Step 9: Save Results

Store evaluation results and baseline metrics for Module 3.

In [24]:
# Save detailed results
results_df.to_csv('evaluation_results.csv', index=False)
print("✓ Saved detailed results to evaluation_results.csv")

# Save baseline metrics
with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2, default=str)
print("✓ Saved baseline metrics to baseline_metrics.json")

# Store for next module
%store baseline_metrics
%store production_thresholds
%store REGION
print("\n✓ Data stored for Module 3!")

✓ Saved detailed results to evaluation_results.csv
✓ Saved baseline metrics to baseline_metrics.json
Stored 'baseline_metrics' (dict)
Stored 'production_thresholds' (dict)
Stored 'REGION' (str)

✓ Data stored for Module 3!


---

## Module 2b Complete!

You have successfully evaluated the single-agent product catalog system using **strands-evals** with the **response caching optimization**.

### Key Optimization

| Approach | Agent Invocations | Time |
|----------|-------------------|------|
| Naive (per evaluator) | 5 cases × 6 evaluators = **30 calls** | ~50 min |
| Optimized (cached) | **5 calls** + 6 cached evaluations | ~10 min |

### Files Created

| File | Description |
|------|-------------|
| `evaluation_results.csv` | Per-case results with all metrics |
| `baseline_metrics.json` | Aggregate baseline metrics |

### Key Observations

- Single agent handles product catalog queries only (Module 1 scope)
- Customer role tested with read-only product tools
- Evaluation metrics adapted for single-agent architecture
- Baseline metrics established for production comparison

### Next Steps

- Compare with **02a-deepeval-evaluation.ipynb** results
- Test admin role with write operations
- Review failing test cases to improve agent prompts
- Use results as baseline for Module 3 production deployment